In [1]:
# C:/IDEAL_Programming/src/train_final_api_models_rf_combined.py
# ============================================================
# FINAL API RANDOM FOREST MODELS - COMBINED SCRIPT
#
# Trains:
#   1) RF with-history
#      - extended lag / rolling features
#      - train-only home statistics
#      - home_id as categorical
#      - sampled training for feasible runtime
#
#   2) RF no-history
#      - simple static + time + weather features
#      - NO home_id
#      - NO lag/history features
#      - NO KNN
#      - NO behavioral profiles
#      - final refit on all data before test
#
# Output:
#   processed/models/final_api_models_rf/
#     ├── training_summary.json
#     ├── with_history/
#     │   ├── model.joblib
#     │   ├── preprocessor.pkl
#     │   ├── feature_config.json
#     │   ├── metadata.json
#     │   ├── home_stats.csv
#     │   ├── regime_thresholds.json
#     │   ├── regime_calibrator.json
#     │   └── test_predictions.csv
#     └── no_history/
#         ├── model.joblib
#         ├── preprocessor.pkl
#         ├── feature_config.json
#         ├── metadata.json
#         ├── test_predictions.csv
#         ├── nohistory_test_comparison.csv
#         └── metrics_by_home.csv
# ============================================================

from __future__ import annotations

import json
import math
import shutil
import time
import warnings
from pathlib import Path
from typing import Any, Dict, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"

OUT_ROOT = BASE_DIR / "processed" / "models" / "final_api_models"/ "RF" 
WITH_HISTORY_DIR = OUT_ROOT / "with_history"
no_history_DIR = OUT_ROOT / "no_history"

OUT_ROOT.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

RANDOM_STATE = 42

TEST_DAYS = 30
VAL_DAYS_TOTAL = 30
CAL_DAYS = 15

BAD_HOMES = ["home171", "home219", "home228", "home271"]
REMOVE_BAD_HOMES = True

TRAIN_WITH_HISTORY = True
TRAIN_no_history = True

CLEAR_WITH_HISTORY_DIR = True
CLEAR_no_history_DIR = True

USE_LOG_TARGET = True

# For RF no-history, this matched the better old setup.
CLIP_TARGET_TRAIN_NOHISTORY = True

# For RF with-history, keep as previous final API RF run.
CLIP_TARGET_TRAIN_HISTORY = False

CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

UNKNOWN_LABEL = "unknown"

STATIC_NUMERIC = [
    "external_temperature",
    "internal_temperature",
    "residents",
    "total_floor_area_m2",
    "num_electric_appliances",
]

TIME_NUMERIC_EXTENDED = [
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
]

# Simple no-history features, same spirit as the original RF script.
NOHISTORY_NUMERIC = [
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
    "external_temperature",
    "total_floor_area_m2",
    "residents",
]

NOHISTORY_CATEGORICAL = [
    "hometype",
    "urban_rural_class",
]

HISTORY_CATEGORICAL = [
    HOME_COL,
    "hometype",
    "urban_rural_class",
    "consumption_regime",
]

LAG_HOURS = [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]
ROLLING_WINDOWS = [3, 6, 12, 24, 48, 168]

REQUIRED_HISTORY_COLS = [
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "lag_336h",
    "roll_mean_24h",
    "roll_mean_168h",
]

MAX_RF_TRAIN_ROWS_WITH_HISTORY = 350_000

# None = full final train for no-history.
MAX_RF_TRAIN_ROWS_no_history = None

RF_WITH_HISTORY_PARAMS = dict(
    n_estimators=120,
    max_depth=20,
    min_samples_leaf=15,
    min_samples_split=30,
    max_features=0.55,
    bootstrap=True,
    max_samples=0.80,
    n_jobs=-1,
    random_state=RANDOM_STATE + 11,
    verbose=1,
)

RF_no_history_PARAMS = dict(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features=0.7,
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)


# ============================================================
# UTILS
# ============================================================

def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def file_size_mb(path: Path) -> float:
    if not path.exists():
        return float("nan")
    return path.stat().st_size / (1024 * 1024)


def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > eps
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": mape_safe(y_true, y_pred),
        "sMAPE_%": smape_safe(y_true, y_pred),
        "bias_Wh": float(np.mean(y_pred - y_true)),
    }


def evaluate_predictions(df_eval: pd.DataFrame, pred_col: str) -> Dict[str, float]:
    y_true = df_eval[TARGET_COL].values
    y_pred = df_eval[pred_col].values

    metrics = compute_metrics(y_true, y_pred)

    daily = (
        df_eval
        .assign(date=df_eval[TIME_COL].dt.date)
        .groupby([HOME_COL, "date"], as_index=False)
        .agg(
            actual_kWh=(TARGET_COL, lambda x: x.sum() / 1000),
            pred_kWh=(pred_col, lambda x: x.sum() / 1000),
        )
    )

    daily["abs_error_kWh"] = np.abs(daily["actual_kWh"] - daily["pred_kWh"])
    daily["abs_error_pct"] = (
        daily["abs_error_kWh"] /
        daily["actual_kWh"].replace(0, np.nan) *
        100
    )

    metrics.update({
        "daily_abs_error_kWh_mean": float(daily["abs_error_kWh"].mean()),
        "daily_abs_error_pct_mean": float(daily["abs_error_pct"].mean()),
        "num_home_days": int(len(daily)),
    })

    return metrics


def print_metrics(title: str, metrics: Dict[str, float]) -> None:
    print("=" * 100)
    print(title)
    print("=" * 100)

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def transform_target(y):
    y = np.asarray(y, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y


def inverse_target(y_hat):
    y_hat = np.asarray(y_hat, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat


def maybe_clip_target(y_train, enabled: bool):
    if not enabled:
        return y_train, None

    lo = np.quantile(y_train, CLIP_Q_LOW)
    hi = np.quantile(y_train, CLIP_Q_HIGH)

    return np.clip(y_train, lo, hi), {
        "low": float(lo),
        "high": float(hi),
    }


def sample_train_for_rf(train_df: pd.DataFrame, max_rows, random_state: int) -> pd.DataFrame:
    if max_rows is None:
        return train_df.copy().reset_index(drop=True)

    if len(train_df) <= max_rows:
        return train_df.copy().reset_index(drop=True)

    frac = max_rows / len(train_df)
    sampled_parts = []

    for _, g in train_df.groupby(HOME_COL, sort=False):
        n = max(1, int(round(len(g) * frac)))
        n = min(n, len(g))
        sampled_parts.append(g.sample(n=n, random_state=random_state))

    sampled = pd.concat(sampled_parts, axis=0)

    if len(sampled) > max_rows:
        sampled = sampled.sample(n=max_rows, random_state=random_state)

    return sampled.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=UNKNOWN_LABEL)),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def benchmark_prediction_speed(model, X, sample_sizes=(24, 168, 720)) -> Dict[str, float]:
    results = {}
    n_available = X.shape[0]

    for n in sample_sizes:
        n_use = min(n, n_available)
        if n_use <= 0:
            continue

        X_sample = X[:n_use]

        start = time.perf_counter()
        _ = model.predict(X_sample)
        elapsed = time.perf_counter() - start

        results[f"predict_seconds_{n_use}_rows"] = float(elapsed)

    return results


# ============================================================
# COMMON FEATURE ENGINEERING
# ============================================================

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["hour"] = df[TIME_COL].dt.hour
    df["day_of_week"] = df[TIME_COL].dt.dayofweek
    df["month"] = df[TIME_COL].dt.month
    df["day_of_month"] = df[TIME_COL].dt.day
    df["week_of_year"] = df[TIME_COL].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    if "is_holiday" not in df.columns:
        df["is_holiday"] = 0

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    return df


def temporal_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    max_time = df[TIME_COL].max()

    test_start = max_time - pd.Timedelta(days=TEST_DAYS) + pd.Timedelta(hours=1)
    val_start = test_start - pd.Timedelta(days=VAL_DAYS_TOTAL)
    cal_start = test_start - pd.Timedelta(days=CAL_DAYS)

    train_df = df[df[TIME_COL] < val_start].copy()
    es_df = df[(df[TIME_COL] >= val_start) & (df[TIME_COL] < cal_start)].copy()
    cal_df = df[(df[TIME_COL] >= cal_start) & (df[TIME_COL] < test_start)].copy()
    test_df = df[df[TIME_COL] >= test_start].copy()

    return train_df, es_df, cal_df, test_df


# ============================================================
# WITH-HISTORY FEATURES
# ============================================================

def add_lag_and_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values([HOME_COL, TIME_COL]).copy()
    g = df.groupby(HOME_COL)[TARGET_COL]

    for lag in LAG_HOURS:
        df[f"lag_{lag}h"] = g.shift(lag)

    shifted = g.shift(1)

    for w in ROLLING_WINDOWS:
        df[f"roll_mean_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .mean()
            .reset_index(level=0, drop=True)
        )

    for w in [24, 48, 168]:
        df[f"roll_std_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .std()
            .reset_index(level=0, drop=True)
        )

        df[f"roll_min_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .min()
            .reset_index(level=0, drop=True)
        )

        df[f"roll_max_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .max()
            .reset_index(level=0, drop=True)
        )

    df["lag_1h_minus_24h"] = df["lag_1h"] - df["lag_24h"]
    df["lag_24h_minus_168h"] = df["lag_24h"] - df["lag_168h"]
    df["lag_48h_minus_168h"] = df["lag_48h"] - df["lag_168h"]
    df["roll_24h_div_168h"] = df["roll_mean_24h"] / (df["roll_mean_168h"] + 1.0)
    df["roll_3h_div_24h"] = df["roll_mean_3h"] / (df["roll_mean_24h"] + 1.0)

    return df


def build_train_only_home_stats(train_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    tmp = train_df.copy()
    tmp["date"] = tmp[TIME_COL].dt.date

    hourly_stats = (
        tmp
        .groupby(HOME_COL)
        .agg(
            home_mean_Wh=(TARGET_COL, "mean"),
            home_median_Wh=(TARGET_COL, "median"),
            home_std_Wh=(TARGET_COL, "std"),
            home_min_Wh=(TARGET_COL, "min"),
            home_max_Wh=(TARGET_COL, "max"),
            home_p75_Wh=(TARGET_COL, lambda x: np.percentile(x, 75)),
            home_p90_Wh=(TARGET_COL, lambda x: np.percentile(x, 90)),
            home_p95_Wh=(TARGET_COL, lambda x: np.percentile(x, 95)),
            home_p99_Wh=(TARGET_COL, lambda x: np.percentile(x, 99)),
        )
        .reset_index()
    )

    daily = (
        tmp
        .groupby([HOME_COL, "date"], as_index=False)
        .agg(daily_kWh=(TARGET_COL, lambda x: x.sum() / 1000))
    )

    daily_stats = (
        daily
        .groupby(HOME_COL)
        .agg(
            home_daily_mean_kWh=("daily_kWh", "mean"),
            home_daily_median_kWh=("daily_kWh", "median"),
            home_daily_std_kWh=("daily_kWh", "std"),
            home_daily_p75_kWh=("daily_kWh", lambda x: np.percentile(x, 75)),
            home_daily_p90_kWh=("daily_kWh", lambda x: np.percentile(x, 90)),
        )
        .reset_index()
    )

    hourly_profile = (
        tmp
        .groupby([HOME_COL, "hour"], as_index=False)
        .agg(home_hour_mean_Wh=(TARGET_COL, "mean"))
    )

    hourly_profile_wide = (
        hourly_profile
        .pivot(index=HOME_COL, columns="hour", values="home_hour_mean_Wh")
        .reset_index()
    )

    hourly_profile_wide.columns = [
        HOME_COL if c == HOME_COL else f"home_hour_{int(c):02d}_mean_Wh"
        for c in hourly_profile_wide.columns
    ]

    home_stats = (
        hourly_stats
        .merge(daily_stats, on=HOME_COL, how="left")
        .merge(hourly_profile_wide, on=HOME_COL, how="left")
    )

    q_low = home_stats["home_daily_mean_kWh"].quantile(0.33)
    q_high = home_stats["home_daily_mean_kWh"].quantile(0.66)
    q_very_high = home_stats["home_daily_mean_kWh"].quantile(0.90)

    def assign_regime(x):
        if pd.isna(x):
            return UNKNOWN_LABEL
        if x <= q_low:
            return "low"
        if x <= q_high:
            return "medium"
        if x <= q_very_high:
            return "high"
        return "high_high"

    home_stats["consumption_regime"] = home_stats["home_daily_mean_kWh"].apply(assign_regime)

    thresholds = {
        "q_low_daily_kWh": float(q_low),
        "q_high_daily_kWh": float(q_high),
        "q_very_high_daily_kWh": float(q_very_high),
    }

    return home_stats, thresholds


def fit_regime_calibrator(cal_eval: pd.DataFrame, pred_col: str) -> Dict[str, Any]:
    params = {}

    for regime, g in cal_eval.groupby("consumption_regime"):
        if len(g) < 500:
            params[regime] = {"type": "none", "scale": 1.0}
            continue

        y_true = g[TARGET_COL].values
        y_pred = g[pred_col].values

        raw_scale = float((np.sum(y_true) + 1e-6) / (np.sum(y_pred) + 1e-6))
        raw_scale = float(np.clip(raw_scale, 0.95, 1.20))

        if regime == "high_high":
            alpha = 0.75
        elif regime in ["high", "medium"]:
            alpha = 0.10
        else:
            alpha = 0.0

        scale = 1.0 + alpha * (raw_scale - 1.0)

        if abs(scale - 1.0) < 1e-9:
            params[regime] = {"type": "none", "scale": 1.0}
        else:
            params[regime] = {
                "type": "scale",
                "scale": float(scale),
                "alpha": alpha,
                "raw_scale": raw_scale,
            }

    for regime in ["low", "medium", "high", "high_high", UNKNOWN_LABEL]:
        params.setdefault(regime, {"type": "none", "scale": 1.0})

    return {
        "type": "regime_specific",
        "regime_params": params,
    }


def apply_regime_calibrator(pred: np.ndarray, regimes: np.ndarray, calibrator: Dict[str, Any]) -> np.ndarray:
    pred = np.asarray(pred).copy()
    regimes = pd.Series(regimes).fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL}).values

    calibrated = pred.copy()

    for regime, p in calibrator["regime_params"].items():
        mask = regimes == regime
        if mask.sum() == 0:
            continue
        if p["type"] == "scale":
            calibrated[mask] = pred[mask] * p["scale"]

    return np.clip(calibrated, 0, None)


# ============================================================
# DATA LOADING
# ============================================================

def load_and_prepare_base_data() -> Tuple[pd.DataFrame, list]:
    print("=" * 100)
    print("Loading dataset")
    print("=" * 100)

    df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
    df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows loaded: {len(df):,}")
    print(f"Homes loaded: {df[HOME_COL].nunique()}")
    print(f"Date range: {df[TIME_COL].min()} to {df[TIME_COL].max()}")

    existing_bad = []

    if REMOVE_BAD_HOMES:
        existing_bad = [h for h in BAD_HOMES if h in set(df[HOME_COL].unique())]

        if existing_bad:
            print("Removing bad homes:", existing_bad)
            df = df[~df[HOME_COL].isin(existing_bad)].copy()

    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[TARGET_COL]).copy()
    df[TARGET_COL] = df[TARGET_COL].clip(lower=0)

    for col in STATIC_NUMERIC:
        if col not in df.columns:
            df[col] = np.nan

    for col in ["hometype", "urban_rural_class"]:
        if col not in df.columns:
            df[col] = UNKNOWN_LABEL

    df = add_time_features(df)

    for c in ["hometype", "urban_rural_class"]:
        df[c] = (
            df[c]
            .fillna(UNKNOWN_LABEL)
            .astype(str)
            .replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})
        )

    for c in STATIC_NUMERIC + TIME_NUMERIC_EXTENDED:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df, existing_bad


# ============================================================
# TRAIN WITH-HISTORY RF
# ============================================================

def train_with_history_model(df_base: pd.DataFrame) -> Dict[str, Any]:
    print("=" * 100)
    print("WITH-HISTORY TRAINING - FINAL RF")
    print("=" * 100)

    if CLEAR_WITH_HISTORY_DIR and WITH_HISTORY_DIR.exists():
        print("Clearing:", WITH_HISTORY_DIR)
        shutil.rmtree(WITH_HISTORY_DIR)

    WITH_HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    df_hist = add_lag_and_rolling_features(df_base)
    df_hist = df_hist.dropna(subset=REQUIRED_HISTORY_COLS).copy()
    df_hist = df_hist.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows after history features: {len(df_hist):,}")
    print(f"Homes after history features: {df_hist[HOME_COL].nunique()}")

    hist_train, hist_es, hist_cal, hist_test = temporal_split(df_hist)

    print("=" * 100)
    print("With-history temporal split")
    print("=" * 100)
    print(f"Train: {hist_train[TIME_COL].min()} to {hist_train[TIME_COL].max()} | rows: {len(hist_train):,}")
    print(f"Early-stop val: {hist_es[TIME_COL].min()} to {hist_es[TIME_COL].max()} | rows: {len(hist_es):,}")
    print(f"Calibration val: {hist_cal[TIME_COL].min()} to {hist_cal[TIME_COL].max()} | rows: {len(hist_cal):,}")
    print(f"Test: {hist_test[TIME_COL].min()} to {hist_test[TIME_COL].max()} | rows: {len(hist_test):,}")

    home_stats, regime_thresholds = build_train_only_home_stats(hist_train)

    home_stats.to_csv(WITH_HISTORY_DIR / "home_stats.csv", index=False)
    save_json(regime_thresholds, WITH_HISTORY_DIR / "regime_thresholds.json")

    hist_train = hist_train.merge(home_stats, on=HOME_COL, how="left")
    hist_es = hist_es.merge(home_stats, on=HOME_COL, how="left")
    hist_cal = hist_cal.merge(home_stats, on=HOME_COL, how="left")
    hist_test = hist_test.merge(home_stats, on=HOME_COL, how="left")

    lag_cols = [c for c in df_hist.columns if c.startswith("lag_")]
    rolling_cols = [c for c in df_hist.columns if c.startswith("roll_")]

    home_numeric_cols = [
        c for c in home_stats.columns
        if c not in [HOME_COL, "consumption_regime"]
        and pd.api.types.is_numeric_dtype(home_stats[c])
    ]

    numeric_cols = [
        c for c in STATIC_NUMERIC + TIME_NUMERIC_EXTENDED + lag_cols + rolling_cols + home_numeric_cols
        if c in hist_train.columns
    ]

    categorical_cols = [
        c for c in HISTORY_CATEGORICAL
        if c in hist_train.columns
    ]

    for split_df in [hist_train, hist_es, hist_cal, hist_test]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")

        for c in categorical_cols:
            split_df[c] = (
                split_df[c]
                .fillna(UNKNOWN_LABEL)
                .astype(str)
                .replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})
            )

    print("=" * 100)
    print("With-history RF features")
    print("=" * 100)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Categorical:", categorical_cols)

    hist_train_model = sample_train_for_rf(
        hist_train,
        max_rows=MAX_RF_TRAIN_ROWS_WITH_HISTORY,
        random_state=RANDOM_STATE + 101,
    )

    print("=" * 100)
    print("RF with-history training sample")
    print("=" * 100)
    print(f"Full train rows: {len(hist_train):,}")
    print(f"RF sampled train rows: {len(hist_train_model):,}")
    print(f"Sampled homes: {hist_train_model[HOME_COL].nunique()}")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    preprocessor.fit(hist_train[numeric_cols + categorical_cols])

    X_train = preprocessor.transform(hist_train_model[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(hist_cal[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(hist_test[numeric_cols + categorical_cols])

    y_train_raw = hist_train_model[TARGET_COL].values
    y_train_used, clip_bounds = maybe_clip_target(
        y_train_raw,
        enabled=CLIP_TARGET_TRAIN_HISTORY,
    )
    y_train = transform_target(y_train_used)

    print("=" * 100)
    print("With-history RF matrices")
    print("=" * 100)
    print(f"X_train: {X_train.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    model = RandomForestRegressor(**RF_WITH_HISTORY_PARAMS)

    print("=" * 100)
    print("Training WITH-HISTORY RF")
    print("=" * 100)

    start_train = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = float(time.perf_counter() - start_train)

    pred_cal = inverse_target(model.predict(X_cal))
    pred_test = inverse_target(model.predict(X_test))

    pred_cal = np.clip(pred_cal, 0, None)
    pred_test = np.clip(pred_test, 0, None)

    cal_eval = hist_cal[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["consumption_regime"] = (
        hist_cal["consumption_regime"]
        .fillna(UNKNOWN_LABEL)
        .astype(str)
        .replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})
        .values
    )
    cal_eval["prediction_Wh"] = pred_cal

    regime_calibrator = fit_regime_calibrator(cal_eval, "prediction_Wh")
    save_json(regime_calibrator, WITH_HISTORY_DIR / "regime_calibrator.json")

    test_eval = hist_test[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["consumption_regime"] = (
        hist_test["consumption_regime"]
        .fillna(UNKNOWN_LABEL)
        .astype(str)
        .replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})
        .values
    )
    test_eval["prediction_Wh"] = pred_test
    test_eval["prediction_peak_daily_Wh"] = apply_regime_calibrator(
        pred_test,
        test_eval["consumption_regime"].values,
        regime_calibrator,
    )

    metrics_balanced = evaluate_predictions(test_eval, "prediction_Wh")
    metrics_peak_daily = evaluate_predictions(test_eval, "prediction_peak_daily_Wh")

    print_metrics("FINAL TEST METRICS - RF WITH HISTORY BALANCED", metrics_balanced)
    print_metrics("FINAL TEST METRICS - RF WITH HISTORY PEAK/DAILY OPTIONAL", metrics_peak_daily)

    model_path = WITH_HISTORY_DIR / "model.joblib"
    preprocessor_path = WITH_HISTORY_DIR / "preprocessor.pkl"
    feature_config_path = WITH_HISTORY_DIR / "feature_config.json"
    metadata_path = WITH_HISTORY_DIR / "metadata.json"
    predictions_path = WITH_HISTORY_DIR / "test_predictions.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)

    prediction_benchmark = benchmark_prediction_speed(model, X_test)

    feature_config = {
        "mode": "with_history",
        "model_family": "RandomForestRegressor",
        "model_name": "rf_with_history_extended_sampled",
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_HISTORY,
        "uses_home_id_as_feature": True,
        "uses_lag_features": True,
        "uses_rolling_features": True,
        "uses_home_stats": True,
        "uses_knn_profiles": False,
        "uses_behavior_profiles": False,
        "required_history_cols": REQUIRED_HISTORY_COLS,
        "lag_hours": LAG_HOURS,
        "rolling_windows": ROLLING_WINDOWS,
        "default_prediction_column": "prediction_Wh",
        "optional_prediction_column": "prediction_peak_daily_Wh",
    }

    save_json(feature_config, feature_config_path)

    metadata = {
        "model_family": "RandomForestRegressor",
        "model_name": "rf_with_history_extended_sampled",
        "mode": "with_history",
        "rf_params": RF_WITH_HISTORY_PARAMS,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_HISTORY,
        "clip_bounds": clip_bounds,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "preprocessor_file_size_mb": file_size_mb(preprocessor_path),
        "prediction_benchmark": prediction_benchmark,
        "full_train_rows": int(len(hist_train)),
        "sampled_train_rows": int(len(hist_train_model)),
        "metrics_balanced": metrics_balanced,
        "metrics_peak_daily_optional": metrics_peak_daily,
        "regime_thresholds": regime_thresholds,
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "home_stats": str(WITH_HISTORY_DIR / "home_stats.csv"),
            "regime_thresholds": str(WITH_HISTORY_DIR / "regime_thresholds.json"),
            "regime_calibrator": str(WITH_HISTORY_DIR / "regime_calibrator.json"),
            "predictions": str(predictions_path),
        },
        "train_period": {
            "start": str(hist_train[TIME_COL].min()),
            "end": str(hist_train[TIME_COL].max()),
            "rows": int(len(hist_train)),
        },
        "early_stop_validation_period_not_used_by_rf": {
            "start": str(hist_es[TIME_COL].min()),
            "end": str(hist_es[TIME_COL].max()),
            "rows": int(len(hist_es)),
        },
        "calibration_validation_period": {
            "start": str(hist_cal[TIME_COL].min()),
            "end": str(hist_cal[TIME_COL].max()),
            "rows": int(len(hist_cal)),
        },
        "test_period": {
            "start": str(hist_test[TIME_COL].min()),
            "end": str(hist_test[TIME_COL].max()),
            "rows": int(len(hist_test)),
        },
    }

    save_json(metadata, metadata_path)
    test_eval.to_csv(predictions_path, index=False)

    return {
        "metrics_balanced": metrics_balanced,
        "metrics_peak_daily_optional": metrics_peak_daily,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "rf_params": RF_WITH_HISTORY_PARAMS,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "prediction_benchmark": prediction_benchmark,
    }


# ============================================================
# TRAIN no-history RF
# ============================================================

def train_no_history_model(df_base: pd.DataFrame) -> Dict[str, Any]:
    print("=" * 100)
    print("no-history TRAINING - RF SIMPLE FINAL REFIT")
    print("=" * 100)

    if CLEAR_no_history_DIR and no_history_DIR.exists():
        print("Clearing:", no_history_DIR)
        shutil.rmtree(no_history_DIR)

    no_history_DIR.mkdir(parents=True, exist_ok=True)

    df_NOHISTORY = df_base.sort_values(TIME_COL).reset_index(drop=True).copy()

    train_df, es_df, cal_df, test_df = temporal_split(df_NOHISTORY)

    final_train_df = (
        pd.concat([train_df, es_df, cal_df], axis=0)
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    print("=" * 100)
    print("no-history temporal split")
    print("=" * 100)
    print(f"Train: {train_df[TIME_COL].min()} to {train_df[TIME_COL].max()} | rows: {len(train_df):,}")
    print(f"Early-stop val: {es_df[TIME_COL].min()} to {es_df[TIME_COL].max()} | rows: {len(es_df):,}")
    print(f"Calibration val: {cal_df[TIME_COL].min()} to {cal_df[TIME_COL].max()} | rows: {len(cal_df):,}")
    print(f"Final train used: {final_train_df[TIME_COL].min()} to {final_train_df[TIME_COL].max()} | rows: {len(final_train_df):,}")
    print(f"Test: {test_df[TIME_COL].min()} to {test_df[TIME_COL].max()} | rows: {len(test_df):,}")

    numeric_cols = [c for c in NOHISTORY_NUMERIC if c in final_train_df.columns]
    categorical_cols = [c for c in NOHISTORY_CATEGORICAL if c in final_train_df.columns]

    for split_df in [final_train_df, train_df, es_df, cal_df, test_df]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")

        for c in categorical_cols:
            split_df[c] = (
                split_df[c]
                .fillna(UNKNOWN_LABEL)
                .astype(str)
                .replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})
            )

    print("=" * 100)
    print("no-history RF simple features")
    print("=" * 100)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Numeric:", numeric_cols)
    print("Categorical:", categorical_cols)

    train_df_model = sample_train_for_rf(
        final_train_df,
        max_rows=MAX_RF_TRAIN_ROWS_no_history,
        random_state=RANDOM_STATE + 202,
    )

    print("=" * 100)
    print("RF no-history final training rows")
    print("=" * 100)
    print(f"Original train rows: {len(train_df):,}")
    print(f"Final train rows including val/cal: {len(final_train_df):,}")
    print(f"Used train rows: {len(train_df_model):,}")
    print(f"Used homes: {train_df_model[HOME_COL].nunique()}")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    preprocessor.fit(final_train_df[numeric_cols + categorical_cols])

    X_train = preprocessor.transform(train_df_model[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(cal_df[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols])

    y_train_raw = train_df_model[TARGET_COL].values
    y_train_used, clip_bounds = maybe_clip_target(
        y_train_raw,
        enabled=CLIP_TARGET_TRAIN_NOHISTORY,
    )
    y_train = transform_target(y_train_used)

    print("=" * 100)
    print("no-history RF matrices")
    print("=" * 100)
    print(f"X_train: {X_train.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    model = RandomForestRegressor(**RF_no_history_PARAMS)

    print("=" * 100)
    print("Training no-history RF SIMPLE FINAL REFIT")
    print("=" * 100)

    start_train = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = float(time.perf_counter() - start_train)

    pred_cal = inverse_target(model.predict(X_cal))
    pred_test = inverse_target(model.predict(X_test))

    pred_cal = np.clip(pred_cal, 0, None)
    pred_test = np.clip(pred_test, 0, None)

    cal_eval = cal_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["prediction_Wh"] = pred_cal

    test_eval = test_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["prediction_Wh"] = pred_test

    metrics_cal_diagnostic = evaluate_predictions(cal_eval, "prediction_Wh")
    metrics_test = evaluate_predictions(test_eval, "prediction_Wh")

    print_metrics(
        "DIAGNOSTIC METRICS ON CALIBRATION PERIOD - RF no-history FINAL REFIT",
        metrics_cal_diagnostic,
    )
    print_metrics("FINAL TEST METRICS - RF no-history FINAL REFIT", metrics_test)

    model_path = no_history_DIR / "model.joblib"
    preprocessor_path = no_history_DIR / "preprocessor.pkl"
    feature_config_path = no_history_DIR / "feature_config.json"
    metadata_path = no_history_DIR / "metadata.json"
    predictions_path = no_history_DIR / "test_predictions.csv"
    comparison_path = no_history_DIR / "nohistory_test_comparison.csv"
    home_metrics_path = no_history_DIR / "metrics_by_home.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)

    prediction_benchmark = benchmark_prediction_speed(model, X_test)

    feature_config = {
        "mode": "no_history",
        "model_family": "RandomForestRegressor",
        "model_name": "rf_no_history_simple_no_knn_final_refit",
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_NOHISTORY,
        "clip_q_low": CLIP_Q_LOW,
        "clip_q_high": CLIP_Q_HIGH,
        "uses_home_id_as_feature": False,
        "uses_lag_features": False,
        "uses_rolling_features": False,
        "uses_home_stats": False,
        "uses_behavior_profiles": False,
        "uses_knn_profiles": False,
        "final_refit_uses_train_es_cal": True,
        "description": (
            "Simple RF no-history model using only time, weather and static household features. "
            "No home_id, no lags, no behavioral profiles, no KNN. "
            "Final model is fitted on all data before the final test period."
        ),
    }

    save_json(feature_config, feature_config_path)

    comparison_df = pd.DataFrame([
        {
            "method": "rf_no_history_simple_no_knn_final_refit",
            **metrics_test,
        }
    ])
    comparison_df.to_csv(comparison_path, index=False)

    home_rows = []

    for home_id, g in test_eval.groupby(HOME_COL):
        m = evaluate_predictions(g, "prediction_Wh")
        home_rows.append({
            HOME_COL: home_id,
            "rows": int(len(g)),
            **m,
        })

    home_metrics_df = pd.DataFrame(home_rows).sort_values("RMSE", ascending=False)
    home_metrics_df.to_csv(home_metrics_path, index=False)

    metadata = {
        "model_family": "RandomForestRegressor",
        "model_name": "rf_no_history_simple_no_knn_final_refit",
        "mode": "no_history",
        "rf_params": RF_no_history_PARAMS,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_NOHISTORY,
        "clip_bounds": clip_bounds,
        "final_refit_uses_train_es_cal": True,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "preprocessor_file_size_mb": file_size_mb(preprocessor_path),
        "prediction_benchmark": prediction_benchmark,
        "full_train_rows_before_final_refit": int(len(train_df)),
        "early_stop_val_rows_used_in_final_refit": int(len(es_df)),
        "calibration_rows_used_in_final_refit": int(len(cal_df)),
        "final_train_rows_used": int(len(final_train_df)),
        "used_train_rows_after_optional_sampling": int(len(train_df_model)),
        "metrics_rf_nohistory_simple_final_refit": metrics_test,
        "calibration_period_metrics_diagnostic_not_out_of_sample": metrics_cal_diagnostic,
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "predictions": str(predictions_path),
            "comparison": str(comparison_path),
            "metrics_by_home": str(home_metrics_path),
        },
        "final_train_period": {
            "start": str(final_train_df[TIME_COL].min()),
            "end": str(final_train_df[TIME_COL].max()),
            "rows": int(len(final_train_df)),
        },
        "test_period": {
            "start": str(test_df[TIME_COL].min()),
            "end": str(test_df[TIME_COL].max()),
            "rows": int(len(test_df)),
        },
    }

    save_json(metadata, metadata_path)
    test_eval.to_csv(predictions_path, index=False)

    return {
        "metrics_rf_nohistory_simple_final_refit": metrics_test,
        "calibration_period_metrics_diagnostic_not_out_of_sample": metrics_cal_diagnostic,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "rf_params": RF_no_history_PARAMS,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "prediction_benchmark": prediction_benchmark,
        "uses_knn_profiles": False,
        "uses_behavior_profiles": False,
        "final_refit_uses_train_es_cal": True,
    }


# ============================================================
# MAIN
# ============================================================

def main():
    df_base, existing_bad = load_and_prepare_base_data()

    summary = {
        "data_path": str(DATA_PATH),
        "target": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "test_days": TEST_DAYS,
        "val_days_total": VAL_DAYS_TOTAL,
        "cal_days": CAL_DAYS,
        "bad_homes_removed": existing_bad,
        "use_log_target": USE_LOG_TARGET,
        "clip_q_low": CLIP_Q_LOW,
        "clip_q_high": CLIP_Q_HIGH,
        "output_root": str(OUT_ROOT),
        "model_selection": {
            "with_history_default": "with_history/model.joblib",
            "no_history_default": "no_history/model.joblib",
            "routing_rule": (
                "Use with_history when recent consumption history is available. "
                "Use no_history when no lag/history consumption values are available."
            ),
        },
    }

    if TRAIN_WITH_HISTORY:
        summary["with_history"] = train_with_history_model(df_base)
    else:
        summary["with_history"] = None

    if TRAIN_no_history:
        summary["no_history"] = train_no_history_model(df_base)
    else:
        summary["no_history"] = None

    save_json(summary, OUT_ROOT / "training_summary.json")

    print("=" * 100)
    print("FINAL API RF MODELS TRAINING COMPLETED")
    print("=" * 100)
    print("OUT_ROOT:", OUT_ROOT)
    print("With-history folder:", WITH_HISTORY_DIR)
    print("no-history folder:", no_history_DIR)
    print("Training summary:", OUT_ROOT / "training_summary.json")

    if summary.get("with_history"):
        print("With-history RF model:", summary["with_history"]["model_path"])
        print("With-history RF size MB:", summary["with_history"].get("model_file_size_mb"))

    if summary.get("no_history"):
        print("no-history RF model:", summary["no_history"]["model_path"])
        print("no-history RF size MB:", summary["no_history"].get("model_file_size_mb"))


if __name__ == "__main__":
    main()

Loading dataset
Rows loaded: 1,529,350
Homes loaded: 254
Date range: 2016-08-10 10:00:00 to 2018-06-30 23:00:00
Removing bad homes: ['home171', 'home219', 'home228', 'home271']
WITH-HISTORY TRAINING - FINAL RF
Rows after history features: 1,429,767
Homes after history features: 250
With-history temporal split
Train: 2016-08-24 10:00:00 to 2018-05-01 23:00:00 | rows: 1,162,265
Early-stop val: 2018-05-02 00:00:00 to 2018-05-16 23:00:00 | rows: 65,853
Calibration val: 2018-05-17 00:00:00 to 2018-05-31 23:00:00 | rows: 72,413
Test: 2018-06-01 00:00:00 to 2018-06-30 23:00:00 | rows: 129,236
With-history RF features
Numeric features: 86
Categorical features: 4
Categorical: ['home_id', 'hometype', 'urban_rural_class', 'consumption_regime']
RF with-history training sample
Full train rows: 1,162,265
RF sampled train rows: 349,997
Sampled homes: 245
With-history RF matrices
X_train: (349997, 340)
X_cal:   (72413, 340)
X_test:  (129236, 340)
Training WITH-HISTORY RF


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  8.9min
[Parallel(n_jobs=-1)]: Done 120 out of 120 | elapsed: 45.8min finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 120 out of 120 | elapsed:    0.4s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 120 out of 120 | elapsed:    0.8s finished


FINAL TEST METRICS - RF WITH HISTORY BALANCED
MAE: 127.9258
RMSE: 302.4671
MAPE_%: 35.9471
sMAPE_%: 32.0478
bias_Wh: -41.6735
daily_abs_error_kWh_mean: 1.1957
daily_abs_error_pct_mean: 15.0754
num_home_days: 5537
FINAL TEST METRICS - RF WITH HISTORY PEAK/DAILY OPTIONAL
MAE: 130.2916
RMSE: 301.1008
MAPE_%: 37.3465
sMAPE_%: 32.6887
bias_Wh: -33.4279
daily_abs_error_kWh_mean: 1.1391
daily_abs_error_pct_mean: 14.7984
num_home_days: 5537


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 120 out of 120 | elapsed:    0.0s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 120 out of 120 | elapsed:    0.0s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 120 out of 120 | elapsed:    0.0s finished


no-history TRAINING - RF SIMPLE FINAL REFIT
no-history temporal split
Train: 2016-08-10 10:00:00 to 2018-05-01 23:00:00 | rows: 1,244,763
Early-stop val: 2018-05-02 00:00:00 to 2018-05-16 23:00:00 | rows: 67,286
Calibration val: 2018-05-17 00:00:00 to 2018-05-31 23:00:00 | rows: 72,482
Final train used: 2016-08-10 10:00:00 to 2018-05-31 23:00:00 | rows: 1,384,531
Test: 2018-06-01 00:00:00 to 2018-06-30 23:00:00 | rows: 129,236
no-history RF simple features
Numeric features: 7
Categorical features: 2
Numeric: ['hour', 'day_of_week', 'is_weekend', 'month', 'external_temperature', 'total_floor_area_m2', 'residents']
Categorical: ['hometype', 'urban_rural_class']
RF no-history final training rows
Original train rows: 1,244,763
Final train rows including val/cal: 1,384,531
Used train rows: 1,384,531
Used homes: 250
no-history RF matrices
X_train: (1384531, 12)
X_cal:   (72482, 12)
X_test:  (129236, 12)
Training no-history RF SIMPLE FINAL REFIT


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:   19.2s
[Parallel(n_jobs=-1)]: Done 152 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.7min finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.5s
[Parallel(n_jobs=24)]: Done 200 out of 200 | elapsed:    0.6s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.9s
[Parallel(n_jobs=24)]: Done 200 out of 200 | elapsed:    1.1s finished


DIAGNOSTIC METRICS ON CALIBRATION PERIOD - RF no-history FINAL REFIT
MAE: 137.3173
RMSE: 312.1873
MAPE_%: 42.5046
sMAPE_%: 35.6039
bias_Wh: -58.0567
daily_abs_error_kWh_mean: 1.8042
daily_abs_error_pct_mean: 23.8923
num_home_days: 3173
FINAL TEST METRICS - RF no-history FINAL REFIT
MAE: 147.5668
RMSE: 319.4000
MAPE_%: 48.2759
sMAPE_%: 39.7932
bias_Wh: -51.8170
daily_abs_error_kWh_mean: 1.9500
daily_abs_error_pct_mean: 27.5647
num_home_days: 5537


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 200 out of 200 | elapsed:    0.0s finished


FINAL API RF MODELS TRAINING COMPLETED
OUT_ROOT: C:\IDEAL_Programming\processed\models\final_api_models\RF
With-history folder: C:\IDEAL_Programming\processed\models\final_api_models\RF\with_history
no-history folder: C:\IDEAL_Programming\processed\models\final_api_models\RF\no_history
Training summary: C:\IDEAL_Programming\processed\models\final_api_models\RF\training_summary.json
With-history RF model: C:\IDEAL_Programming\processed\models\final_api_models\RF\with_history\model.joblib
With-history RF size MB: 146.73341464996338
no-history RF model: C:\IDEAL_Programming\processed\models\final_api_models\RF\no_history\model.joblib
no-history RF size MB: 1151.9368906021118
